# Dyania MASH Risk Stratification - Full Code Notebook

**Purpose:** standalone, notebook-first code submission for the Dyania hackathon case study.  
**Clinical target:** pre-test probability of clinically significant MASH/fibrosis in CKM-enriched adults.  
**Prototype data:** NHANES 2017-March 2020 public-use files with liver ultrasound transient elastography.

This version embeds the core pipeline code directly in notebook cells. It is longer than the narrative notebook, but easier for judges to run if they only inspect one `.ipynb` file.


## Notebook Roadmap

1. Design decisions and leakage rules.
2. Full code definitions split into logical cells.
3. Optional end-to-end NHANES pipeline run.
4. Cohort, performance, thresholds, sensitivity analysis, and clinical-note demo outputs.

The full run may take several minutes because it performs model search, calibration, and bootstrap confidence intervals. By default, the notebook reads precomputed artifacts.


## Design Decisions

| Decision | Rationale |
|---|---|
| MASH/fibrosis-first target | The Dyania deliverable is early MASH risk stratification. CKM is used to enrich and explain risk. |
| Structured-first predictors | Routine data can run upstream of hepatology, FibroScan, or trial screening. |
| Label-leakage control | FibroScan kPa, CAP, biopsy, imaging-confirmed fibrosis, and explicit MASH/fibrosis text are labels or adjudication evidence, not predictors. |
| FIB-4/APRI/NFS as features and baselines | These encode clinical reasoning and create fair rule-based comparators. |
| XGBoost primary model | Captures nonlinear tabular interactions and supports SHAP explanations. |
| ElasticNet benchmark | Gives transparent standardized odds ratios for clinical sanity checks. |
| Smoking included | Available in NHANES and clinically plausible as a fibrosis-risk amplifier. |
| Calibrated probabilities | Risk percentages are shown only after calibration. |


## Imports, configuration, and calibration result class


In [ ]:
"""NHANES MASH/fibrosis risk proof-of-concept pipeline.

This script downloads NHANES 2017-March 2020 public-use XPT files, builds a
CKM-enriched labeled cohort with complete liver elastography, trains clinical
baselines plus ElasticNet and XGBoost models, and writes model artifacts.
"""

from __future__ import annotations

import argparse
import json
import math
from dataclasses import dataclass
from pathlib import Path
import re
from typing import Iterable
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
from scipy.special import expit, logit
from scipy.stats import bootstrap as scipy_bootstrap
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

try:
    from lightgbm import LGBMClassifier
except ImportError:  # pragma: no cover - optional dependency
    LGBMClassifier = None

try:
    from catboost import CatBoostClassifier
except ImportError:  # pragma: no cover - optional dependency
    CatBoostClassifier = None


try:
    ROOT = Path(__file__).resolve().parents[1]
except NameError:
    ROOT = Path.cwd()
    if ROOT.name == "notebooks":
        ROOT = ROOT.parent
RAW_DIR = ROOT / "data" / "raw" / "nhanes"
PROCESSED_DIR = ROOT / "data" / "processed"
ARTIFACT_DIR = ROOT / "artifacts"

NHANES_BASE = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles"
NHANES_FILES = {
    "P_DEMO": f"{NHANES_BASE}/P_DEMO.XPT",
    "P_LUX": f"{NHANES_BASE}/P_LUX.XPT",
    "P_BMX": f"{NHANES_BASE}/P_BMX.XPT",
    "P_BPXO": f"{NHANES_BASE}/P_BPXO.XPT",
    "P_BIOPRO": f"{NHANES_BASE}/P_BIOPRO.XPT",
    "P_CBC": f"{NHANES_BASE}/P_CBC.XPT",
    "P_GHB": f"{NHANES_BASE}/P_GHB.XPT",
    "P_HDL": f"{NHANES_BASE}/P_HDL.XPT",
    "P_TCHOL": f"{NHANES_BASE}/P_TCHOL.XPT",
    "P_TRIGLY": f"{NHANES_BASE}/P_TRIGLY.XPT",
    "P_DIQ": f"{NHANES_BASE}/P_DIQ.XPT",
    "P_BPQ": f"{NHANES_BASE}/P_BPQ.XPT",
    "P_MCQ": f"{NHANES_BASE}/P_MCQ.XPT",
    "P_ALQ": f"{NHANES_BASE}/P_ALQ.XPT",
    "P_SMQ": f"{NHANES_BASE}/P_SMQ.XPT",
    "P_RXQ_RX": f"{NHANES_BASE}/P_RXQ_RX.XPT",
}

REQUIRED_LABEL_COLUMNS = ["SEQN", "LUAXSTAT", "LUXSMED", "LUXCAPM"]


@dataclass
class CalibrationResult:
    method: str
    model: object

    def predict(self, probs: np.ndarray) -> np.ndarray:
        probs = np.asarray(probs, dtype=float)
        if self.method == "identity":
            return np.clip(probs, 1e-6, 1 - 1e-6)
        if self.method == "isotonic":
            return np.clip(np.asarray(self.model.predict(probs), dtype=float), 1e-6, 1 - 1e-6)
        clipped = np.clip(probs, 1e-6, 1 - 1e-6)
        logits = logit(clipped).reshape(-1, 1)
        return np.clip(self.model.predict_proba(logits)[:, 1], 1e-6, 1 - 1e-6)


## Data download, XPT reading, and low-level cleaning helpers


In [ ]:
def ensure_dirs() -> None:
    for directory in (RAW_DIR, PROCESSED_DIR, ARTIFACT_DIR):
        directory.mkdir(parents=True, exist_ok=True)


def download_nhanes_files(force: bool = False) -> None:
    ensure_dirs()
    for name, url in NHANES_FILES.items():
        target = RAW_DIR / f"{name}.XPT"
        if target.exists() and not force:
            continue
        print(f"Downloading {name} from {url}")
        urlretrieve(url, target)


def read_xpt(name: str, columns: Iterable[str] | None = None) -> pd.DataFrame:
    path = RAW_DIR / f"{name}.XPT"
    df = pd.read_sas(path, format="xport")
    if columns is not None:
        available = [col for col in columns if col in df.columns]
        missing = sorted(set(columns) - set(available))
        if missing:
            print(f"{name}: missing columns ignored: {missing}")
        df = df[available]
    return df


def decode_object_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.select_dtypes(include=["object", "str"]).columns:
        out[col] = out[col].map(
            lambda value: value.decode("utf-8", "ignore").strip()
            if isinstance(value, (bytes, bytearray))
            else value
        )
    return out


def first_available(df: pd.DataFrame, candidates: list[str]) -> pd.Series:
    for col in candidates:
        if col in df.columns:
            return df[col]
    return pd.Series(np.nan, index=df.index)


def recode_special_missing(series: pd.Series, missing_codes: set[int | float]) -> pd.Series:
    """Recode variable-specific NHANES missing codes without blanket 7/9 removal."""
    out = pd.to_numeric(series, errors="coerce").copy()
    return out.mask(out.isin(missing_codes), np.nan)


def valid_range(series: pd.Series, low: float | None = None, high: float | None = None) -> pd.Series:
    out = pd.to_numeric(series, errors="coerce")
    if low is not None:
        out = out.mask(out < low, np.nan)
    if high is not None:
        out = out.mask(out > high, np.nan)
    return out


def alcohol_frequency_days_per_year(code: pd.Series) -> pd.Series:
    """Map NHANES ALQ121 frequency codes to approximate drinking days/year."""
    numeric = recode_special_missing(code, {77, 99})
    mapping = {
        0: 0.0,
        1: 365.0,
        2: 300.0,
        3: 3.5 * 52.0,
        4: 2.0 * 52.0,
        5: 52.0,
        6: 2.5 * 12.0,
        7: 12.0,
        8: 9.0,
        9: 4.5,
        10: 1.5,
    }
    return numeric.map(mapping)


def estimate_alcohol_drinks_per_week(alq121: pd.Series, alq130: pd.Series) -> pd.Series:
    days_per_year = alcohol_frequency_days_per_year(alq121)
    drinks_per_day = recode_special_missing(alq130, {777, 999})
    drinks_per_day = drinks_per_day.replace(15, 15.0)
    return (days_per_year * drinks_per_day) / 52.0


def mean_of_available(df: pd.DataFrame, columns: list[str]) -> pd.Series:
    available = [col for col in columns if col in df.columns]
    if not available:
        return pd.Series(np.nan, index=df.index)
    return df[available].apply(pd.to_numeric, errors="coerce").mean(axis=1, skipna=True)


## Clinical feature engineering helpers


In [ ]:
def add_liver_scores(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    age = pd.to_numeric(out.get("age"), errors="coerce")
    ast = valid_range(out.get("ast"), 1, 2000)
    alt = valid_range(out.get("alt"), 1, 2000)
    platelets = valid_range(out.get("platelets"), 1, 1000)
    bmi = pd.to_numeric(out.get("bmi"), errors="coerce")
    albumin = pd.to_numeric(out.get("albumin"), errors="coerce")
    diabetes = pd.to_numeric(
        out.get("diabetes", pd.Series(0, index=out.index)), errors="coerce"
    ).fillna(0)

    out["fib4"] = (age * ast) / (platelets * np.sqrt(alt))
    out["apri"] = ((ast / 40.0) / platelets) * 100.0
    out["ast_alt_ratio"] = ast / alt
    out["nfs"] = (
        -1.675
        + 0.037 * age
        + 0.094 * bmi
        + 1.13 * diabetes
        + 0.99 * out["ast_alt_ratio"]
        - 0.013 * platelets
        - 0.66 * albumin
    )
    for col in ["fib4", "apri", "ast_alt_ratio", "nfs"]:
        out[col] = out[col].replace([np.inf, -np.inf], np.nan)
    return out


def calculate_egfr_2021(scr: pd.Series, age: pd.Series, female: pd.Series) -> pd.Series:
    scr = valid_range(scr, 0.1, 20)
    age = valid_range(age, 18, 120)
    female = pd.to_numeric(female, errors="coerce").fillna(0).astype(int)
    kappa = np.where(female == 1, 0.7, 0.9)
    alpha = np.where(female == 1, -0.241, -0.302)
    egfr = 142 * (np.minimum(scr / kappa, 1) ** alpha) * (np.maximum(scr / kappa, 1) ** -1.200)
    egfr = egfr * (0.9938**age) * np.where(female == 1, 1.012, 1.0)
    return pd.Series(egfr, index=scr.index).replace([np.inf, -np.inf], np.nan)


def medication_text_columns(rx: pd.DataFrame) -> pd.Series:
    text_cols = [col for col in ["RXDDRUG", "RXDRSD1", "RXDRSD2", "RXDRSD3"] if col in rx.columns]
    if not text_cols:
        return pd.Series("", index=rx.index)
    return (
        rx[text_cols]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
        .str.lower()
    )


def aggregate_medications(rx: pd.DataFrame) -> pd.DataFrame:
    rx = decode_object_columns(rx)
    if "SEQN" not in rx.columns:
        return pd.DataFrame(columns=["SEQN", "med_diabetes", "med_lipid", "med_hypertension"])

    text = medication_text_columns(rx)
    diabetes_pat = re.compile(
        r"metformin|insulin|semaglutide|liraglutide|dulaglutide|exenatide|"
        r"empagliflozin|dapagliflozin|canagliflozin|sitagliptin|linagliptin|"
        r"glipizide|glyburide|glimepiride|pioglitazone|diabetes|diabetic"
    )
    lipid_pat = re.compile(
        r"atorvastatin|rosuvastatin|simvastatin|pravastatin|lovastatin|"
        r"pitavastatin|ezetimibe|fenofibrate|gemfibrozil|omega|cholesterol|lipid|triglyceride"
    )
    htn_pat = re.compile(
        r"lisinopril|losartan|valsartan|amlodipine|hydrochlorothiazide|"
        r"chlorthalidone|metoprolol|atenolol|carvedilol|propranolol|"
        r"furosemide|spironolactone|diltiazem|verapamil|hypertension|blood pressure"
    )

    flags = pd.DataFrame(
        {
            "SEQN": rx["SEQN"],
            "med_diabetes": text.str.contains(diabetes_pat).astype(int),
            "med_lipid": text.str.contains(lipid_pat).astype(int),
            "med_hypertension": text.str.contains(htn_pat).astype(int),
            "med_any_count": 1,
        }
    )
    return flags.groupby("SEQN", as_index=False).max()


## NHANES harmonization and cohort-level feature construction


In [ ]:
def load_and_harmonize() -> pd.DataFrame:
    demo = read_xpt("P_DEMO", ["SEQN", "RIDAGEYR", "RIAGENDR", "RIDRETH3", "WTMECPRP", "SDMVPSU", "SDMVSTRA"])
    lux = read_xpt("P_LUX")
    bmx = read_xpt("P_BMX", ["SEQN", "BMXBMI", "BMXWAIST"])
    bpxo = read_xpt("P_BPXO")
    biopro = read_xpt("P_BIOPRO")
    cbc = read_xpt("P_CBC")
    ghb = read_xpt("P_GHB")
    hdl = read_xpt("P_HDL")
    tchol = read_xpt("P_TCHOL")
    trigly = read_xpt("P_TRIGLY")
    diq = read_xpt("P_DIQ")
    bpq = read_xpt("P_BPQ")
    mcq = read_xpt("P_MCQ")
    alq = read_xpt("P_ALQ")
    smq = read_xpt("P_SMQ", ["SEQN", "SMQ020", "SMQ040", "SMD030", "SMD641", "SMD650"])
    rx = read_xpt("P_RXQ_RX")

    merged = demo.copy()
    for df in [lux, bmx, bpxo, biopro, cbc, ghb, hdl, tchol, trigly, diq, bpq, mcq, alq, smq]:
        if "SEQN" in df.columns:
            merged = merged.merge(df, on="SEQN", how="left")
    merged = merged.merge(aggregate_medications(rx), on="SEQN", how="left")

    out = pd.DataFrame({"SEQN": merged["SEQN"]})
    out["age"] = valid_range(merged["RIDAGEYR"], 0, 120)
    out["female"] = (merged["RIAGENDR"] == 2).astype(int)
    out["sex_code"] = merged["RIAGENDR"]
    out["race_ethnicity_code"] = merged.get("RIDRETH3")
    out["sample_weight"] = merged.get("WTMECPRP")
    out["sdmvpsu"] = merged.get("SDMVPSU")
    out["sdmvstra"] = merged.get("SDMVSTRA")

    out["lux_complete"] = (merged.get("LUAXSTAT") == 1).astype(int) if "LUAXSTAT" in merged else 0
    out["liver_stiffness_kpa"] = valid_range(merged.get("LUXSMED"), 0, 100)
    out["cap_dbm"] = valid_range(merged.get("LUXCAPM"), 0, 500)
    out["lux_iqr_median"] = valid_range(first_available(merged, ["LUXSIQRM", "LUXSIQR"]), 0, 100)

    out["bmi"] = valid_range(merged.get("BMXBMI"), 10, 90)
    out["waist_cm"] = valid_range(merged.get("BMXWAIST"), 30, 250)
    out["sbp_mean"] = valid_range(mean_of_available(merged, ["BPXOSY2", "BPXOSY3", "BPXOSY1"]), 50, 260)
    out["dbp_mean"] = valid_range(mean_of_available(merged, ["BPXODI2", "BPXODI3", "BPXODI1"]), 20, 160)

    out["albumin"] = valid_range(first_available(merged, ["LBXSAL"]), 1, 8)
    out["bilirubin"] = valid_range(first_available(merged, ["LBXSTB", "LBXTB"]), 0, 50)
    out["alp"] = valid_range(first_available(merged, ["LBXSAPSI", "LBXSAP"]), 1, 3000)
    out["ast"] = valid_range(first_available(merged, ["LBXSASSI", "LBXSAST"]), 1, 2000)
    out["alt"] = valid_range(first_available(merged, ["LBXSATSI", "LBXSALT"]), 1, 2000)
    out["creatinine"] = valid_range(first_available(merged, ["LBXSCR"]), 0.1, 20)
    out["glucose"] = valid_range(first_available(merged, ["LBXSGL", "LBXGLU"]), 20, 1000)
    out["platelets"] = valid_range(first_available(merged, ["LBXPLTSI", "LBXPLT"]), 1, 1000)
    out["hemoglobin"] = valid_range(first_available(merged, ["LBXHGB"]), 3, 25)
    out["hba1c"] = valid_range(first_available(merged, ["LBXGH"]), 2, 20)
    out["hdl"] = valid_range(first_available(merged, ["LBDHDD", "LBDHDL"]), 1, 300)
    out["total_chol"] = valid_range(first_available(merged, ["LBXTC"]), 20, 1000)
    out["triglycerides"] = valid_range(first_available(merged, ["LBXTR"]), 1, 5000)
    out["ldl"] = valid_range(first_available(merged, ["LBDLDL"]), 1, 1000)
    out["alcohol_drinks_per_week"] = estimate_alcohol_drinks_per_week(
        merged.get("ALQ121", pd.Series(np.nan, index=merged.index)),
        merged.get("ALQ130", pd.Series(np.nan, index=merged.index)),
    )
    out["high_alcohol_use"] = (
        ((out["female"] == 1) & (out["alcohol_drinks_per_week"] > 14))
        | ((out["female"] == 0) & (out["alcohol_drinks_per_week"] > 21))
    ).astype(int)

    smq020 = recode_special_missing(merged.get("SMQ020", pd.Series(np.nan, index=merged.index)), {7, 9})
    smq040 = recode_special_missing(merged.get("SMQ040", pd.Series(np.nan, index=merged.index)), {7, 9})
    out["ever_smoker"] = (smq020 == 1).astype(int)
    out["never_smoker"] = (smq020 == 2).astype(int)
    out["current_smoker"] = ((smq020 == 1) & smq040.isin([1, 2])).astype(int)
    out["former_smoker"] = ((smq020 == 1) & (smq040 == 3)).astype(int)
    out["smoking_age_started"] = valid_range(first_available(merged, ["SMD030"]), 5, 90)
    out["smoking_days_past_30"] = valid_range(first_available(merged, ["SMD641"]), 0, 30)
    out["current_cigarettes_per_day"] = valid_range(first_available(merged, ["SMD650"]), 0, 120)

    out["med_diabetes"] = merged.get("med_diabetes", 0).fillna(0).astype(int)
    out["med_lipid"] = merged.get("med_lipid", 0).fillna(0).astype(int)
    out["med_hypertension"] = merged.get("med_hypertension", 0).fillna(0).astype(int)
    out["med_any_count"] = merged.get("med_any_count", 0).fillna(0).astype(int)

    diabetes_questionnaire = recode_special_missing(merged.get("DIQ010", pd.Series(np.nan, index=merged.index)), {7, 9})
    out["diabetes_questionnaire"] = (diabetes_questionnaire == 1).astype(int)
    out["diabetes"] = (
        (out["diabetes_questionnaire"] == 1)
        | (out["hba1c"] >= 6.5)
        | (out["glucose"] >= 200)
        | (out["med_diabetes"] == 1)
    ).astype(int)
    out["prediabetes"] = ((out["hba1c"].between(5.7, 6.4, inclusive="both")) & (out["diabetes"] == 0)).astype(int)

    bpq020 = recode_special_missing(merged.get("BPQ020", pd.Series(np.nan, index=merged.index)), {7, 9})
    out["hypertension"] = (
        (bpq020 == 1)
        | (out["sbp_mean"] >= 130)
        | (out["dbp_mean"] >= 80)
        | (out["med_hypertension"] == 1)
    ).astype(int)

    bpq080 = recode_special_missing(merged.get("BPQ080", pd.Series(np.nan, index=merged.index)), {7, 9})
    low_hdl = np.where(out["female"] == 1, out["hdl"] < 50, out["hdl"] < 40)
    out["dyslipidemia"] = (
        (bpq080 == 1)
        | (out["total_chol"] >= 200)
        | (out["triglycerides"] >= 150)
        | low_hdl
        | (out["ldl"] >= 130)
        | (out["med_lipid"] == 1)
    ).astype(int)

    out["obesity"] = (out["bmi"] >= 30).astype(int)
    out["egfr"] = calculate_egfr_2021(out["creatinine"], out["age"], out["female"])
    out["ckd"] = (out["egfr"] < 60).astype(int)
    cvd_cols = [col for col in ["MCQ160B", "MCQ160C", "MCQ160D", "MCQ160E", "MCQ160F"] if col in merged.columns]
    if cvd_cols:
        cvd_data = merged[cvd_cols].apply(lambda s: recode_special_missing(s, {7, 9}))
        out["cvd"] = (cvd_data == 1).any(axis=1).astype(int)
    else:
        out["cvd"] = 0

    out["metabolic_syndrome_proxy"] = (
        out[["obesity", "hypertension", "dyslipidemia"]].sum(axis=1)
        + ((out["diabetes"] == 1) | (out["prediabetes"] == 1)).astype(int)
    )
    out["metabolic_syndrome_proxy"] = (out["metabolic_syndrome_proxy"] >= 3).astype(int)
    out["ckm_burden_count"] = out[["diabetes", "obesity", "hypertension", "dyslipidemia", "ckd", "cvd"]].sum(axis=1)
    out["ckm_primary"] = ((out["diabetes"] == 1) | (out["ckm_burden_count"] >= 2)).astype(int)
    out["ckm_any"] = (out["ckm_burden_count"] >= 1).astype(int)

    out = add_liver_scores(out)
    out["label_lsm_ge_8"] = ((out["lux_complete"] == 1) & (out["liver_stiffness_kpa"] >= 8.0)).astype(int)
    out["label_lsm_ge_12"] = ((out["lux_complete"] == 1) & (out["liver_stiffness_kpa"] >= 12.0)).astype(int)
    out["steatosis_cap_ge_274"] = (out["cap_dbm"] >= 274).astype(int)

    for col in model_base_features(out):
        out[f"{col}_missing"] = out[col].isna().astype(int)

    return out


## Model feature list, validation split, and candidate model definitions


In [ ]:
def model_base_features(df: pd.DataFrame | None = None) -> list[str]:
    return [
        "age",
        "female",
        "race_ethnicity_code",
        "bmi",
        "waist_cm",
        "sbp_mean",
        "dbp_mean",
        "albumin",
        "bilirubin",
        "alp",
        "ast",
        "alt",
        "creatinine",
        "egfr",
        "glucose",
        "platelets",
        "hemoglobin",
        "hba1c",
        "hdl",
        "total_chol",
        "triglycerides",
        "ldl",
        "ever_smoker",
        "never_smoker",
        "current_smoker",
        "former_smoker",
        "smoking_age_started",
        "smoking_days_past_30",
        "current_cigarettes_per_day",
        "diabetes",
        "prediabetes",
        "obesity",
        "hypertension",
        "dyslipidemia",
        "ckd",
        "cvd",
        "metabolic_syndrome_proxy",
        "ckm_burden_count",
        "med_diabetes",
        "med_lipid",
        "med_hypertension",
        "med_any_count",
        "fib4",
        "apri",
        "nfs",
        "ast_alt_ratio",
    ]


def model_features(df: pd.DataFrame) -> list[str]:
    base = [col for col in model_base_features() if col in df.columns]
    missing = [f"{col}_missing" for col in model_base_features() if f"{col}_missing" in df.columns]
    return base + missing


def choose_validation_design(n_labeled: int, n_positive: int) -> dict[str, object]:
    if n_labeled >= 2000 and n_positive >= 200:
        return {"name": "train_validation_test", "train": 0.60, "validation": 0.20, "test": 0.20}
    return {"name": "development_test_cv_calibration", "development": 0.80, "test": 0.20}


def split_data(X: pd.DataFrame, y: pd.Series, design: dict[str, object], seed: int):
    if design["name"] == "train_validation_test":
        X_train, X_temp, y_train, y_temp = train_test_split(
            X, y, test_size=0.40, stratify=y, random_state=seed
        )
        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=seed
        )
        return X_train, X_val, X_test, y_train, y_val, y_test
    X_dev, X_test, y_dev, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=seed
    )
    return X_dev, None, X_test, y_dev, None, y_test


def make_elasticnet(seed: int) -> Pipeline:
    numeric_preprocess = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=False)),
            ("scaler", StandardScaler()),
        ]
    )
    return Pipeline(
        steps=[
            ("prep", ColumnTransformer([("num", numeric_preprocess, slice(0, None))])),
            (
                "model",
                LogisticRegressionCV(
                    Cs=[0.01, 0.1, 1.0, 10.0],
                    cv=5,
                    penalty="elasticnet",
                    solver="saga",
                    l1_ratios=[0.2, 0.5, 0.8],
                    scoring="roc_auc",
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=seed,
                    n_jobs=-1,
                ),
            ),
        ]
    )


def make_xgb_grid(y_train: pd.Series, seed: int) -> GridSearchCV:
    positives = max(int(y_train.sum()), 1)
    negatives = max(int(len(y_train) - y_train.sum()), 1)
    scale_pos_weight = negatives / positives
    estimator = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=seed,
        n_jobs=1,
        scale_pos_weight=scale_pos_weight,
        missing=np.nan,
    )
    param_grid = {
        "n_estimators": [120, 240],
        "max_depth": [2, 3],
        "learning_rate": [0.03, 0.08],
        "min_child_weight": [1, 5],
        "subsample": [0.85],
        "colsample_bytree": [0.85],
        "reg_lambda": [1.0],
    }
    return GridSearchCV(
        estimator,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=seed),
        n_jobs=1,
        refit=True,
    )


def make_lgbm_grid(y_train: pd.Series, seed: int) -> GridSearchCV | None:
    if LGBMClassifier is None:
        return None
    positives = max(int(y_train.sum()), 1)
    negatives = max(int(len(y_train) - y_train.sum()), 1)
    scale_pos_weight = negatives / positives
    estimator = LGBMClassifier(
        objective="binary",
        random_state=seed,
        n_jobs=1,
        scale_pos_weight=scale_pos_weight,
        verbosity=-1,
    )
    param_grid = {
        "n_estimators": [120, 240],
        "num_leaves": [7, 15, 31],
        "learning_rate": [0.03, 0.08],
        "min_child_samples": [20, 50],
        "subsample": [0.85],
        "colsample_bytree": [0.85],
        "reg_lambda": [1.0],
    }
    return GridSearchCV(
        estimator,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=seed),
        n_jobs=1,
        refit=True,
    )


def make_catboost_grid(y_train: pd.Series, seed: int) -> GridSearchCV | None:
    if CatBoostClassifier is None:
        return None
    estimator = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
        thread_count=1,
        auto_class_weights="Balanced",
    )
    param_grid = {
        "iterations": [120, 240],
        "depth": [2, 3, 4],
        "learning_rate": [0.03, 0.08],
        "l2_leaf_reg": [3.0, 10.0],
    }
    return GridSearchCV(
        estimator,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=seed),
        n_jobs=1,
        refit=True,
    )


## Calibration, thresholds, metrics, and decision-curve utilities


In [ ]:
def fit_calibrator(y_true: pd.Series, raw_probs: np.ndarray) -> CalibrationResult:
    raw_probs = np.clip(np.asarray(raw_probs, dtype=float), 1e-6, 1 - 1e-6)
    y_true = np.asarray(y_true, dtype=int)
    positives = int(y_true.sum())
    if len(y_true) >= 500 and positives >= 50:
        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(raw_probs, y_true)
        return CalibrationResult("isotonic", iso)
    if len(np.unique(y_true)) < 2:
        return CalibrationResult("identity", None)
    sigmoid = LogisticRegression(solver="lbfgs")
    sigmoid.fit(logit(raw_probs).reshape(-1, 1), y_true)
    return CalibrationResult("sigmoid", sigmoid)


def calibrated_fit_predict(estimator, X_train, y_train, X_val, y_val, X_test, design_name: str):
    if design_name == "train_validation_test":
        estimator.fit(X_train, y_train)
        raw_val = estimator.predict_proba(X_val)[:, 1]
        calibrator = fit_calibrator(y_val, raw_val)
        val_probs = calibrator.predict(raw_val)
        raw_test = estimator.predict_proba(X_test)[:, 1]
        return estimator, calibrator.predict(raw_test), calibrator.method, val_probs

    method = "isotonic" if int(y_train.sum()) >= 50 and len(y_train) >= 500 else "sigmoid"
    try:
        calibrated = CalibratedClassifierCV(estimator=clone(estimator), method=method, cv=5)
    except TypeError:
        calibrated = CalibratedClassifierCV(base_estimator=clone(estimator), method=method, cv=5)
    calibrated.fit(X_train, y_train)
    train_probs = calibrated.predict_proba(X_train)[:, 1]
    return calibrated, calibrated.predict_proba(X_test)[:, 1], f"cv_{method}", train_probs


def threshold_metrics(y_true: np.ndarray, probs: np.ndarray, threshold: float = 0.5) -> dict[str, float]:
    pred = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    ppv = tp / (tp + fp) if (tp + fp) else np.nan
    npv = tn / (tn + fn) if (tn + fn) else np.nan
    return {
        "threshold": threshold,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "ppv": ppv,
        "npv": npv,
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }


def choose_action_thresholds(y_true: np.ndarray, probs: np.ndarray) -> dict[str, float]:
    precision, recall, thresholds = precision_recall_curve(y_true, probs)
    # Low-risk cutoff: highest threshold that keeps sensitivity around 90%.
    low_candidates = thresholds[recall[:-1] >= 0.90]
    low = float(np.max(low_candidates)) if len(low_candidates) else float(np.quantile(probs, 0.25))
    # High-risk cutoff: threshold with best F1 among remaining thresholds.
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-9)
    high = float(thresholds[int(np.nanargmax(f1))]) if len(thresholds) else 0.5
    if high <= low:
        high = float(np.quantile(probs, 0.80))
    return {"low_intermediate": low, "intermediate_high": high}


def decision_curve(y_true, y_prob, thresholds: Iterable[float]) -> pd.DataFrame:
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    n = len(y_true)
    prevalence = y_true.mean() if n else 0
    rows = []
    for pt in thresholds:
        pred = y_prob >= pt
        tp = int(((pred == 1) & (y_true == 1)).sum())
        fp = int(((pred == 1) & (y_true == 0)).sum())
        harm = pt / (1 - pt)
        model_nb = (tp / n) - (fp / n) * harm
        treat_all_nb = prevalence - (1 - prevalence) * harm
        rows.append(
            {
                "threshold": pt,
                "model_net_benefit": model_nb,
                "treat_all_net_benefit": treat_all_nb,
                "treat_none_net_benefit": 0.0,
            }
        )
    return pd.DataFrame(rows)


def net_benefit_values(y_true, y_prob, thresholds: Iterable[float]) -> list[float]:
    return decision_curve(y_true, y_prob, thresholds)["model_net_benefit"].tolist()


def safe_metric(func, y_true, scores) -> float:
    try:
        return float(func(y_true, scores))
    except ValueError:
        return float("nan")


def evaluate_predictions(y_true: pd.Series, probs: np.ndarray, thresholds: dict[str, float]) -> dict[str, object]:
    y_arr = np.asarray(y_true, dtype=int)
    probs = np.clip(np.asarray(probs, dtype=float), 1e-6, 1 - 1e-6)
    brier = brier_score_loss(y_arr, probs)
    calib_model = LogisticRegression(solver="lbfgs")
    if len(np.unique(y_arr)) == 2:
        calib_model.fit(logit(probs).reshape(-1, 1), y_arr)
        calib_intercept = float(calib_model.intercept_[0])
        calib_slope = float(calib_model.coef_[0][0])
    else:
        calib_intercept = float("nan")
        calib_slope = float("nan")
    return {
        "auroc": safe_metric(roc_auc_score, y_arr, probs),
        "auprc": safe_metric(average_precision_score, y_arr, probs),
        "brier": float(brier),
        "calibration_intercept": calib_intercept,
        "calibration_slope": calib_slope,
        "low_cutoff": thresholds["low_intermediate"],
        "high_cutoff": thresholds["intermediate_high"],
        "metrics_at_high_cutoff": threshold_metrics(y_arr, probs, thresholds["intermediate_high"]),
        "metrics_at_0_5": threshold_metrics(y_arr, probs, 0.5),
    }


def bootstrap_ci(y_true, probs, n_bootstrap: int, seed: int) -> dict[str, dict[str, float]]:
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true, dtype=int)
    probs = np.asarray(probs, dtype=float)
    rows = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2:
            continue
        rows.append(
            {
                "auroc": roc_auc_score(y_true[idx], probs[idx]),
                "auprc": average_precision_score(y_true[idx], probs[idx]),
                "brier": brier_score_loss(y_true[idx], probs[idx]),
            }
        )
    if not rows:
        return {}
    boot = pd.DataFrame(rows)
    return {
        col: {
            "mean": float(boot[col].mean()),
            "ci_low": float(boot[col].quantile(0.025)),
            "ci_high": float(boot[col].quantile(0.975)),
        }
        for col in boot.columns
    }


## Clinical baselines, cohort summaries, patient cards, and artifact writers


In [ ]:
def baseline_scores(df: pd.DataFrame) -> dict[str, pd.Series]:
    return {
        "fib4": df["fib4"],
        "apri": df["apri"],
        "nfs": df["nfs"],
    }


def calibrate_baseline_score(
    score_val: pd.Series,
    y_val: pd.Series,
    score_test: pd.Series,
) -> tuple[np.ndarray, str]:
    """Convert a raw clinical score into calibrated test-set probabilities.

    FIB-4/APRI/NFS are useful ordinal risk scores, but Brier score, decision
    curves, and action thresholds require probabilities. A one-variable
    logistic calibration keeps the baseline clinically recognizable while
    making those probability-based comparisons mathematically valid.
    """
    val = pd.to_numeric(score_val, errors="coerce")
    test = pd.to_numeric(score_test, errors="coerce")
    y = pd.Series(y_val).astype(int)

    clean = val.notna() & y.notna()
    if clean.sum() >= 4 and y[clean].nunique() == 2:
        impute_value = float(val[clean].median())
        model = LogisticRegression(solver="lbfgs")
        model.fit(val[clean].to_numpy().reshape(-1, 1), y[clean].to_numpy())
        probs = model.predict_proba(test.fillna(impute_value).to_numpy().reshape(-1, 1))[:, 1]
        return np.clip(probs, 1e-6, 1 - 1e-6), "sigmoid"

    if val.notna().any():
        impute_value = float(val.median())
        combined = pd.concat([val, test.fillna(impute_value)], ignore_index=True)
        ranks = combined.rank(pct=True).iloc[len(val):].to_numpy()
        return np.clip(ranks, 1e-6, 1 - 1e-6), "rank_fallback"

    return np.full(len(test), 0.5), "uninformative_fallback"


def summarize_cohort(df: pd.DataFrame, cohort: pd.DataFrame) -> dict[str, object]:
    weighted_prev = np.nan
    weights = pd.to_numeric(cohort.get("sample_weight"), errors="coerce")
    if weights.notna().any() and weights.sum() > 0:
        weighted_prev = float(np.average(cohort["label_lsm_ge_8"], weights=weights.fillna(0)))
    adult_complete_ckm = (
        (df["age"] >= 18)
        & (df["lux_complete"] == 1)
        & df["liver_stiffness_kpa"].notna()
        & (df["ckm_primary"] == 1)
    )
    med_agreement = {
        "diabetes_questionnaire_or_lab": int(((cohort["diabetes_questionnaire"] == 1) | (cohort["hba1c"] >= 6.5)).sum()),
        "diabetes_med_flag": int((cohort["med_diabetes"] == 1).sum()),
        "diabetes_combined": int((cohort["diabetes"] == 1).sum()),
        "med_diabetes_without_questionnaire_diabetes": int(
            ((cohort["med_diabetes"] == 1) & (cohort["diabetes_questionnaire"] == 0)).sum()
        ),
    }
    missingness = cohort[model_features(cohort)].isna().mean().sort_values(ascending=False).to_dict()
    return {
        "total_rows_loaded": int(len(df)),
        "total_adults": int((df["age"] >= 18).sum()),
        "adults_with_complete_elastography": int(((df["age"] >= 18) & (df["lux_complete"] == 1) & df["liver_stiffness_kpa"].notna()).sum()),
        "ckm_primary_labeled_before_alcohol_exclusion_n": int(adult_complete_ckm.sum()),
        "high_alcohol_excluded_from_primary_cohort_n": int((adult_complete_ckm & (df["high_alcohol_use"] == 1)).sum()),
        "ckm_primary_labeled_n": int(len(cohort)),
        "positive_lsm_ge_8_n": int(cohort["label_lsm_ge_8"].sum()),
        "positive_lsm_ge_12_n": int(cohort["label_lsm_ge_12"].sum()),
        "unweighted_lsm_ge_8_prevalence": float(cohort["label_lsm_ge_8"].mean()),
        "weighted_lsm_ge_8_prevalence": weighted_prev,
        "ckm_any_labeled_n": int(((df["age"] >= 18) & (df["lux_complete"] == 1) & df["liver_stiffness_kpa"].notna() & (df["ckm_any"] == 1)).sum()),
        "medication_phenotype_agreement": med_agreement,
        "top_missingness": {k: float(v) for k, v in list(missingness.items())[:20]},
    }


def save_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, allow_nan=True), encoding="utf-8")


def risk_card(model_name: str, model, X_test: pd.DataFrame, y_test: pd.Series, probs: np.ndarray, features: list[str]) -> dict[str, object]:
    probs = np.asarray(probs, dtype=float)
    y_arr = np.asarray(y_test, dtype=int)
    positive_positions = np.flatnonzero(y_arr == 1)
    if len(positive_positions):
        idx = int(positive_positions[np.argmax(probs[positive_positions])])
    else:
        idx = int(np.argmax(probs))
    row = X_test.iloc[[idx]]
    risk = float(probs[idx])
    card = {
        "model": model_name,
        "risk_score_0_100": round(risk * 100, 1),
        "observed_label_lsm_ge_8": int(y_test.iloc[idx]),
        "recommended_next_step": "FibroScan / hepatology referral" if risk >= 0.5 else "complete labs / repeat risk assessment",
        "note": "Feature contributions are approximate and intended for demonstration.",
    }
    try:
        import shap

        if hasattr(model, "best_estimator_"):
            estimator = model.best_estimator_
        elif hasattr(model, "calibrated_classifiers_"):
            estimator = model.calibrated_classifiers_[0].estimator
        else:
            estimator = model
        if isinstance(estimator, XGBClassifier):
            explainer = shap.TreeExplainer(estimator)
            shap_values = explainer.shap_values(row)
            vals = np.asarray(shap_values)[0]
            top = np.argsort(np.abs(vals))[::-1][:8]
            card["top_contributors_raw_shap"] = [
                {"feature": features[i], "value": float(row.iloc[0, i]), "raw_shap": float(vals[i])}
                for i in top
            ]
    except Exception as exc:
        card["top_contributors_raw_shap_error"] = str(exc)
    return card


def secondary_endpoint_metrics(
    labeled: pd.DataFrame,
    X_test: pd.DataFrame,
    xgb_probs: np.ndarray,
    elastic_probs: np.ndarray,
) -> dict[str, object]:
    y_secondary = labeled.loc[X_test.index, "label_lsm_ge_12"].astype(int)
    out = {
        "endpoint": "liver_stiffness_kpa >= 12.0",
        "positive_n_test": int(y_secondary.sum()),
        "test_n": int(len(y_secondary)),
        "note": "Models are trained on the sensitive >=8.0 kPa endpoint; this evaluates whether the same scores enrich the stricter >=12.0 kPa endpoint.",
        "xgboost_auroc": safe_metric(roc_auc_score, y_secondary, xgb_probs),
        "xgboost_auprc": safe_metric(average_precision_score, y_secondary, xgb_probs),
        "elasticnet_auroc": safe_metric(roc_auc_score, y_secondary, elastic_probs),
        "elasticnet_auprc": safe_metric(average_precision_score, y_secondary, elastic_probs),
    }
    for name, score in baseline_scores(labeled.loc[X_test.index]).items():
        clean = score.notna()
        out[f"{name}_raw_auroc"] = safe_metric(roc_auc_score, y_secondary[clean], score[clean])
        out[f"{name}_raw_auprc"] = safe_metric(average_precision_score, y_secondary[clean], score[clean])
    return out


def write_test_predictions(
    labeled: pd.DataFrame,
    X_test: pd.DataFrame,
    xgb_probs: np.ndarray,
    elastic_probs: np.ndarray,
    xgb_thresholds: dict[str, float],
) -> None:
    cols = [
        "SEQN",
        "liver_stiffness_kpa",
        "cap_dbm",
        "label_lsm_ge_8",
        "label_lsm_ge_12",
        "fib4",
        "apri",
        "nfs",
        "age",
        "female",
        "bmi",
        "waist_cm",
        "hba1c",
        "diabetes",
        "ckm_burden_count",
    ]
    available = [col for col in cols if col in labeled.columns]
    out = labeled.loc[X_test.index, available].copy()
    out["xgboost_probability_lsm_ge_8"] = np.asarray(xgb_probs, dtype=float)
    out["elasticnet_probability_lsm_ge_8"] = np.asarray(elastic_probs, dtype=float)
    out["xgboost_risk_tier"] = np.select(
        [
            out["xgboost_probability_lsm_ge_8"] < xgb_thresholds["low_intermediate"],
            out["xgboost_probability_lsm_ge_8"] >= xgb_thresholds["intermediate_high"],
        ],
        ["low", "high"],
        default="intermediate",
    )
    out["xgboost_false_positive_at_high_cutoff"] = (
        (out["xgboost_risk_tier"] == "high") & (out["label_lsm_ge_8"] == 0)
    ).astype(int)
    out.sort_values("xgboost_probability_lsm_ge_8", ascending=False).to_csv(
        ARTIFACT_DIR / "test_predictions.csv", index=False
    )


def save_model_selection_artifact(
    design: dict[str, object],
    xgb_grid: GridSearchCV,
    lgbm_grid: GridSearchCV | None,
    catboost_grid: GridSearchCV | None,
    xgb_thresholds: dict[str, float],
    metrics: dict[str, object],
) -> None:
    artifact = {
        "selection_principle": (
            "Candidate models are tuned on training/validation data; the held-out "
            "test set is reserved for final reporting, not threshold or hyperparameter selection."
        ),
        "validation_design": design,
        "candidate_models": [
            "FIB-4 calibrated baseline",
            "APRI calibrated baseline",
            "NAFLD Fibrosis Score calibrated baseline",
            "ElasticNet logistic regression",
            "XGBoost gradient-boosted trees",
            "LightGBM gradient-boosted trees",
            "CatBoost gradient-boosted trees",
            "Sample-weighted XGBoost sensitivity run",
        ],
        "primary_selection_criteria": [
            "calibration",
            "AUROC",
            "AUPRC",
            "Brier score",
            "sensitivity/NPV at validation-selected action thresholds",
            "Decision Curve Analysis net benefit",
            "clinical interpretability",
        ],
        "xgboost_search_grid": {
            key: [float(v) if isinstance(v, np.floating) else int(v) if isinstance(v, np.integer) else v for v in values]
            for key, values in xgb_grid.param_grid.items()
        },
        "xgboost_best_params": {
            key: float(value) if isinstance(value, np.floating) else int(value) if isinstance(value, np.integer) else value
            for key, value in xgb_grid.best_params_.items()
        },
        "xgboost_best_cv_auroc": float(xgb_grid.best_score_),
        "xgboost_validation_selected_thresholds": xgb_thresholds,
        "lightgbm_search_grid": (
            {
                key: [
                    float(v) if isinstance(v, np.floating) else int(v) if isinstance(v, np.integer) else v
                    for v in values
                ]
                for key, values in lgbm_grid.param_grid.items()
            }
            if lgbm_grid is not None
            else None
        ),
        "lightgbm_best_params": (
            {
                key: float(value) if isinstance(value, np.floating) else int(value) if isinstance(value, np.integer) else value
                for key, value in lgbm_grid.best_params_.items()
            }
            if lgbm_grid is not None
            else None
        ),
        "lightgbm_best_cv_auroc": float(lgbm_grid.best_score_) if lgbm_grid is not None else None,
        "catboost_search_grid": (
            {
                key: [
                    float(v) if isinstance(v, np.floating) else int(v) if isinstance(v, np.integer) else v
                    for v in values
                ]
                for key, values in catboost_grid.param_grid.items()
            }
            if catboost_grid is not None
            else None
        ),
        "catboost_best_params": (
            {
                key: float(value) if isinstance(value, np.floating) else int(value) if isinstance(value, np.integer) else value
                for key, value in catboost_grid.best_params_.items()
            }
            if catboost_grid is not None
            else None
        ),
        "catboost_best_cv_auroc": float(catboost_grid.best_score_) if catboost_grid is not None else None,
        "final_test_metrics_summary": {
            "xgboost": metrics.get("xgboost", {}),
            "lightgbm": metrics.get("lightgbm", {}),
            "catboost": metrics.get("catboost", {}),
            "elasticnet": metrics.get("elasticnet", {}),
            "baselines": metrics.get("baselines", {}),
            "secondary_endpoint_lsm_ge_12": metrics.get("secondary_endpoint_lsm_ge_12", {}),
        },
    }
    save_json(ARTIFACT_DIR / "model_selection.json", artifact)


## End-to-end pipeline runner


In [ ]:
def run_pipeline(force_download: bool, bootstrap_n: int, seed: int) -> dict[str, object]:
    ensure_dirs()
    download_nhanes_files(force=force_download)
    df = load_and_harmonize()
    df.to_csv(PROCESSED_DIR / "nhanes_mash_model.csv", index=False)

    labeled = df[
        (df["age"] >= 18)
        & (df["lux_complete"] == 1)
        & df["liver_stiffness_kpa"].notna()
        & (df["ckm_primary"] == 1)
        & (df["high_alcohol_use"] == 0)
    ].copy()
    summary = summarize_cohort(df, labeled)
    design = choose_validation_design(summary["ckm_primary_labeled_n"], summary["positive_lsm_ge_8_n"])
    summary["validation_design"] = design
    save_json(ARTIFACT_DIR / "cohort_summary.json", summary)

    features = model_features(labeled)
    X = labeled[features].apply(pd.to_numeric, errors="coerce")
    y = labeled["label_lsm_ge_8"].astype(int)
    X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y, design, seed)
    y_threshold_reference = y_val if y_val is not None else y_train

    metrics: dict[str, object] = {"cohort_summary": summary, "feature_count": len(features)}

    baseline_metrics = {}
    baseline_probabilities = {}
    baseline_reference = labeled.loc[X_val.index if X_val is not None else X_train.index]
    y_baseline_reference = y_val if y_val is not None else y_train
    test_baselines = baseline_scores(labeled.loc[X_test.index])
    for name, score_test in test_baselines.items():
        score_ref = baseline_scores(baseline_reference)[name]
        clean = score_test.notna()
        if clean.sum() and len(np.unique(y_test[clean])) == 2:
            probs, method = calibrate_baseline_score(score_ref, y_baseline_reference, score_test)
            ref_probs, _ = calibrate_baseline_score(score_ref, y_baseline_reference, score_ref)
            baseline_probabilities[name] = probs
            thresholds = choose_action_thresholds(y_baseline_reference, ref_probs)
            baseline_metrics[name] = evaluate_predictions(y_test, probs, thresholds)
            baseline_metrics[name]["probability_calibration_method"] = method
            baseline_metrics[name]["raw_auroc"] = safe_metric(roc_auc_score, y_test[clean], score_test[clean])
            baseline_metrics[name]["raw_auprc"] = safe_metric(
                average_precision_score, y_test[clean], score_test[clean]
            )
    metrics["baselines"] = baseline_metrics

    elastic = make_elasticnet(seed)
    elastic_model, elastic_probs, elastic_cal, elastic_ref_probs = calibrated_fit_predict(
        elastic, X_train, y_train, X_val, y_val, X_test, design["name"]
    )
    elastic_thresholds = choose_action_thresholds(y_threshold_reference, elastic_ref_probs)
    metrics["elasticnet"] = evaluate_predictions(y_test, elastic_probs, elastic_thresholds)
    metrics["elasticnet"]["calibration_method"] = elastic_cal

    xgb_grid = make_xgb_grid(y_train, seed)
    xgb_grid.fit(X_train, y_train)
    xgb_model, xgb_probs, xgb_cal, xgb_ref_probs = calibrated_fit_predict(
        xgb_grid.best_estimator_, X_train, y_train, X_val, y_val, X_test, design["name"]
    )
    xgb_thresholds = choose_action_thresholds(y_threshold_reference, xgb_ref_probs)
    metrics["xgboost"] = evaluate_predictions(y_test, xgb_probs, xgb_thresholds)
    metrics["xgboost"]["best_params"] = xgb_grid.best_params_
    metrics["xgboost"]["calibration_method"] = xgb_cal
    metrics["xgboost"]["bootstrap_ci"] = bootstrap_ci(y_test, xgb_probs, bootstrap_n, seed)

    lgbm_grid = make_lgbm_grid(y_train, seed)
    lgbm_probs = None
    if lgbm_grid is not None:
        lgbm_grid.fit(X_train, y_train)
        lgbm_model, lgbm_probs, lgbm_cal, lgbm_ref_probs = calibrated_fit_predict(
            lgbm_grid.best_estimator_, X_train, y_train, X_val, y_val, X_test, design["name"]
        )
        lgbm_thresholds = choose_action_thresholds(y_threshold_reference, lgbm_ref_probs)
        metrics["lightgbm"] = evaluate_predictions(y_test, lgbm_probs, lgbm_thresholds)
        metrics["lightgbm"]["best_params"] = lgbm_grid.best_params_
        metrics["lightgbm"]["calibration_method"] = lgbm_cal
    else:
        metrics["lightgbm"] = {"skipped": "lightgbm is not installed"}

    catboost_grid = make_catboost_grid(y_train, seed)
    catboost_probs = None
    if catboost_grid is not None:
        catboost_grid.fit(X_train, y_train)
        catboost_model, catboost_probs, catboost_cal, catboost_ref_probs = calibrated_fit_predict(
            catboost_grid.best_estimator_, X_train, y_train, X_val, y_val, X_test, design["name"]
        )
        catboost_thresholds = choose_action_thresholds(y_threshold_reference, catboost_ref_probs)
        metrics["catboost"] = evaluate_predictions(y_test, catboost_probs, catboost_thresholds)
        metrics["catboost"]["best_params"] = catboost_grid.best_params_
        metrics["catboost"]["calibration_method"] = catboost_cal
    else:
        metrics["catboost"] = {"skipped": "catboost is not installed"}

    metrics["secondary_endpoint_lsm_ge_12"] = secondary_endpoint_metrics(
        labeled, X_test, xgb_probs, elastic_probs
    )

    if X_val is not None and "sample_weight" in labeled.columns:
        w_train = pd.to_numeric(labeled.loc[X_train.index, "sample_weight"], errors="coerce").fillna(1)
        weighted_xgb = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=seed,
            n_jobs=1,
            missing=np.nan,
            scale_pos_weight=max(int((y_train == 0).sum()), 1) / max(int((y_train == 1).sum()), 1),
            **xgb_grid.best_params_,
        )
        weighted_xgb.fit(X_train, y_train, sample_weight=w_train)
        weighted_val = weighted_xgb.predict_proba(X_val)[:, 1]
        weighted_calibrator = fit_calibrator(y_val, weighted_val)
        weighted_ref_probs = weighted_calibrator.predict(weighted_val)
        weighted_probs = weighted_calibrator.predict(weighted_xgb.predict_proba(X_test)[:, 1])
        weighted_thresholds = choose_action_thresholds(y_threshold_reference, weighted_ref_probs)
        metrics["xgboost_weighted_sensitivity"] = evaluate_predictions(y_test, weighted_probs, weighted_thresholds)
        metrics["xgboost_weighted_sensitivity"]["calibration_method"] = weighted_calibrator.method

    dca_thresholds = np.round(np.arange(0.05, 0.51, 0.05), 2)
    dca = decision_curve(y_test, xgb_probs, dca_thresholds)
    dca.to_csv(ARTIFACT_DIR / "decision_curve_xgboost.csv", index=False)
    dca_compare = pd.DataFrame({"threshold": dca_thresholds})
    dca_compare["xgboost"] = net_benefit_values(y_test, xgb_probs, dca_thresholds)
    if lgbm_probs is not None:
        dca_compare["lightgbm"] = net_benefit_values(y_test, lgbm_probs, dca_thresholds)
    if catboost_probs is not None:
        dca_compare["catboost"] = net_benefit_values(y_test, catboost_probs, dca_thresholds)
    dca_compare["elasticnet"] = net_benefit_values(y_test, elastic_probs, dca_thresholds)
    for name, probs in baseline_probabilities.items():
        dca_compare[name] = net_benefit_values(y_test, probs, dca_thresholds)
    dca_compare["treat_all"] = dca["treat_all_net_benefit"]
    dca_compare["treat_none"] = 0.0
    dca_compare.to_csv(ARTIFACT_DIR / "decision_curve_comparison.csv", index=False)

    if hasattr(xgb_model, "feature_importances_"):
        importances = pd.DataFrame({"feature": features, "importance": xgb_model.feature_importances_})
    elif hasattr(xgb_model, "calibrated_classifiers_"):
        estimator = xgb_model.calibrated_classifiers_[0].estimator
        importances = pd.DataFrame({"feature": features, "importance": estimator.feature_importances_})
    else:
        importances = pd.DataFrame({"feature": features, "importance": np.nan})
    importances.sort_values("importance", ascending=False).to_csv(ARTIFACT_DIR / "feature_importance_xgboost.csv", index=False)

    card = risk_card("xgboost", xgb_model, X_test, y_test.reset_index(drop=True), xgb_probs, features)
    save_json(ARTIFACT_DIR / "patient_risk_card.json", card)
    write_test_predictions(labeled, X_test, xgb_probs, elastic_probs, xgb_thresholds)
    save_json(ARTIFACT_DIR / "metrics.json", metrics)
    save_model_selection_artifact(design, xgb_grid, lgbm_grid, catboost_grid, xgb_thresholds, metrics)

    return metrics


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--force-download", action="store_true", help="Re-download NHANES XPT files.")
    parser.add_argument("--bootstrap", type=int, default=500, help="Bootstrap replicates for final test metrics.")
    parser.add_argument("--seed", type=int, default=42, help="Random seed.")
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    metrics = run_pipeline(force_download=args.force_download, bootstrap_n=args.bootstrap, seed=args.seed)
    print(json.dumps(metrics["cohort_summary"], indent=2, allow_nan=True))
    print(json.dumps({"elasticnet": metrics["elasticnet"], "xgboost": metrics["xgboost"]}, indent=2, allow_nan=True))


# In the notebook, call run_pipeline(...) from the execution cell below.


## Run or Load the Pipeline

Set `RUN_FULL_PIPELINE = True` to download/harmonize NHANES and retrain the models from source files. Leave it `False` for quick review of the submitted artifacts.


In [ ]:
RUN_FULL_PIPELINE = False

if RUN_FULL_PIPELINE:
    metrics = run_pipeline(force_download=False, bootstrap_n=300, seed=2026)
else:
    metrics = json.loads((ARTIFACT_DIR / "metrics.json").read_text(encoding="utf-8"))
    print("Loaded precomputed metrics. Set RUN_FULL_PIPELINE=True to recompute from NHANES.")

metrics["xgboost"]


## Cohort Flow


| Step | N / value |
|---|---:|
| Total NHANES rows loaded | 15560 |
| Adults >=18 | 9693 |
| Adults with complete elastography | 7768 |
| CKM-primary labeled before alcohol exclusion | 4827 |
| High-alcohol exclusions | 182 |
| Final CKM-primary labeled cohort | 4645 |
| LSM >=8.0 kPa positives | 646 |
| LSM >=12.0 kPa positives | 222 |


In [ ]:
cohort_summary = json.loads((ARTIFACT_DIR / "cohort_summary.json").read_text(encoding="utf-8"))
pd.Series(cohort_summary).to_frame("value")


## Final XGBoost Performance


| Metric | XGBoost result |
|---|---:|
| AUROC | 0.804 |
| AUPRC | 0.444 |
| Brier score | 0.098 |
| Calibration intercept | 0.000 |
| Calibration slope | 1.031 |
| Low/intermediate threshold | 0.083 |
| Intermediate/high threshold | 0.244 |


## Model Comparison

| model | auroc | auprc | brier | calibration_intercept | calibration_slope | high_risk_cutoff | high_risk_sensitivity | specificity_at_high_cutoff | ppv_at_high_cutoff | npv_at_high_cutoff | test_tp | test_fp | test_tn | test_fn | net_benefit_at_0_20 | net_benefit_at_0_25 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| FIB4 | 0.569 | 0.197 | 0.119 | -1.520 | 0.168 | 0.154 | 0.279 | 0.779 | 0.169 | 0.870 | 36.000 | 177.000 | 623.000 | 93.000 | 0.004 | 0.005 |
| APRI | 0.605 | 0.246 | 0.115 | -0.480 | 0.743 | 0.154 | 0.310 | 0.841 | 0.240 | 0.883 | 40.000 | 127.000 | 673.000 | 89.000 | 0.009 | 0.010 |
| NFS | 0.697 | 0.278 | 0.112 | 0.291 | 1.162 | 0.194 | 0.419 | 0.848 | 0.307 | 0.900 | 54.000 | 122.000 | 678.000 | 75.000 | 0.021 | 0.009 |
| ElasticNet | 0.803 | 0.379 | 0.100 | -0.358 | 0.750 | 0.235 | 0.674 | 0.775 | 0.326 | 0.937 | 87.000 | 180.000 | 620.000 | 42.000 | 0.045 | 0.025 |
| XGBoost | 0.804 | 0.444 | 0.098 | 0.000 | 1.031 | 0.244 | 0.667 | 0.784 | 0.332 | 0.936 | 86.000 | 173.000 | 627.000 | 43.000 | 0.046 | 0.035 |
| LightGBM | 0.802 | 0.464 | 0.094 | 0.063 | 1.074 | 0.281 | 0.481 | 0.901 | 0.440 | 0.915 | 62.000 | 79.000 | 721.000 | 67.000 | 0.045 | 0.039 |
| CatBoost | 0.805 | 0.432 | 0.098 | 0.091 | 1.052 | 0.238 | 0.620 | 0.809 | 0.343 | 0.930 | 80.000 | 153.000 | 647.000 | 49.000 | 0.045 | 0.034 |
| Weighted XGBoost sensitivity | 0.791 | 0.403 | 0.099 | 0.071 | 1.062 | 0.316 | 0.434 | 0.912 | 0.444 | 0.909 | 56.000 | 70.000 | 730.000 | 73.000 |  |  |


## Risk Tiers

| tier_or_decision_rule | n | positive_n | negative_n | observed_positive_rate | mean_predicted_risk | sensitivity | specificity | ppv | npv | tp | fp | tn | fn |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Low tier | 318 | 7 | 311 | 0.022 | 0.015 |  |  |  |  |  |  |  |  |
| Intermediate tier | 352 | 36 | 316 | 0.102 | 0.118 |  |  |  |  |  |  |  |  |
| High tier | 259 | 86 | 173 | 0.332 | 0.334 |  |  |  |  |  |  |  |  |
| Intermediate or high | 611 | 122 | 489 | 0.200 | 0.210 |  |  |  |  |  |  |  |  |
| Decision: escalate above low cutoff | 929 | 129 | 800 | 0.139 | 0.143 | 0.946 | 0.389 | 0.200 | 0.978 | 122.000 | 489.000 | 311.000 | 7.000 |
| Decision: high-risk direct referral | 929 | 129 | 800 | 0.139 | 0.143 | 0.667 | 0.784 | 0.332 | 0.936 | 86.000 | 173.000 | 627.000 | 43.000 |


# Model Metric Interpretation

| Metric | Better Direction | How to read it in this project |
| --- | --- | --- |
| AUROC | Higher | Ranking ability across all possible thresholds. 0.5 is random; 1.0 is perfect. Useful for model discrimination, but not enough alone for clinical deployment. |
| AUPRC | Higher | Precision-recall performance. More informative than AUROC when positives are less common; rewards finding true positives without too many false positives. |
| Brier score | Lower | Mean squared error of predicted probabilities. A lower Brier score means the probability itself is more accurate. |
| Calibration intercept | Closer to 0 | Whether predictions are systematically too high or too low. Positive means underprediction on average; negative means overprediction on average. |
| Calibration slope | Closer to 1 | Whether probabilities are too extreme or too compressed. Less than 1 suggests overconfidence; greater than 1 suggests underconfident spread. |
| High-risk sensitivity | Higher, within acceptable workload | Among true LSM >= 8.0 cases, the fraction assigned to the high-risk action tier. This measures how many true cases get immediate escalation. |
| Specificity at high cutoff | Higher | Among true negatives, the fraction not assigned to the high-risk tier. Higher specificity means fewer unnecessary escalations. |
| PPV at high cutoff | Higher | Among patients flagged high-risk, the fraction truly positive. Useful for referral yield and specialist workload. |
| NPV at high cutoff | Higher | Among patients not flagged high-risk, the fraction truly negative. Useful for safety of not escalating immediately. |
| Net benefit / DCA | Higher | Clinical utility after accounting for false-positive harm at a chosen risk threshold. Compare against treat-all, treat-none, and FIB-4 alone. |


## Same-Model Sensitivity Analysis

# Same-Model Sensitivity Analysis

All rows use the locked XGBoost predictions from the same held-out test patients. The model is not refit for subgroup or endpoint analyses.

| analysis | endpoint | threshold | n | positive_n | observed_positive_rate | mean_predicted_risk | sensitivity | specificity | ppv | npv | tp | fp | tn | fn |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Primary test set: direct high-risk referral | label_lsm_ge_8 | 0.244 | 929 | 129 | 0.139 | 0.143 | 0.667 | 0.784 | 0.332 | 0.936 | 86 | 173 | 627 | 43 |
| Primary test set: escalate above low cutoff | label_lsm_ge_8 | 0.083 | 929 | 129 | 0.139 | 0.143 | 0.946 | 0.389 | 0.200 | 0.978 | 122 | 489 | 311 | 7 |
| Stricter endpoint: LSM >=12 on same patients | label_lsm_ge_12 | 0.244 | 929 | 49 | 0.053 | 0.143 | 0.755 | 0.748 | 0.143 | 0.982 | 37 | 222 | 658 | 12 |
| Diabetes subgroup | label_lsm_ge_8 | 0.244 | 278 | 64 | 0.230 | 0.227 | 0.828 | 0.598 | 0.381 | 0.921 | 53 | 86 | 128 | 11 |
| Obesity plus diabetes subgroup | label_lsm_ge_8 | 0.244 | 156 | 49 | 0.314 | 0.285 | 0.898 | 0.430 | 0.419 | 0.902 | 44 | 61 | 46 | 5 |
| High CKM burden subgroup (>=3) | label_lsm_ge_8 | 0.244 | 526 | 97 | 0.184 | 0.185 | 0.722 | 0.690 | 0.345 | 0.916 | 70 | 133 | 296 | 27 |
| Ever-smoker subgroup | label_lsm_ge_8 | 0.244 | 401 | 54 | 0.135 | 0.147 | 0.630 | 0.761 | 0.291 | 0.930 | 34 | 83 | 264 | 20 |
| Never-smoker subgroup | label_lsm_ge_8 | 0.244 | 527 | 75 | 0.142 | 0.140 | 0.693 | 0.801 | 0.366 | 0.940 | 52 | 90 | 362 | 23 |


## Incidence by Predicted-Risk Quartile

| predicted_risk_quartile | n | observed_positive_n | observed_lsm_ge_8_rate | mean_predicted_risk | min_predicted_risk | max_predicted_risk |
| --- | --- | --- | --- | --- | --- | --- |
| Q1 lowest | 233 | 4 | 0.017 | 0.015 | 0.000 | 0.017 |
| Q2 | 232 | 16 | 0.069 | 0.071 | 0.017 | 0.128 |
| Q3 | 232 | 29 | 0.125 | 0.143 | 0.128 | 0.244 |
| Q4 highest | 232 | 80 | 0.345 | 0.345 | 0.244 | 1.000 |


In [ ]:
try:
    from IPython.display import Image, display
    display(Image(filename=str(ARTIFACT_DIR / "risk_quartile_incidence.png")))
except Exception:
    print(ARTIFACT_DIR / "risk_quartile_incidence.png")


## Clinical-Note Demo

Synthetic Dyania notes are used for abstraction and audit. Explicit MASH, FibroScan, biopsy, and fibrosis-stage text is not used as a predictor. Smoking is extracted as a structured input because it is also present in the NHANES derivation pipeline.


| note_index | source_file | Age | Female sex | BMI | Systolic BP | Diastolic BP | Albumin | Bilirubin | ALP | AST | ALT | Glucose | Creatinine | eGFR | Platelets | Hemoglobin | HbA1c | HDL | Total cholesterol | Triglycerides | LDL | Ever smoker | Never smoker | Current smoker | Former smoker | Type 2 diabetes | Obesity | Hypertension | Dyslipidemia | CKM burden count | FIB-4 | APRI | NAFLD Fibrosis Score | AST/ALT ratio | documented_lsm_kpa | documented_fibrosis_stage | documented_fib4_today | biopsy_mentions_f4 | smoking_status | sedentary_work_note_only | family_history_liver_disease_note_only | clinical_signs_advanced_liver_disease_note_only | note_only_variables_not_model_inputs |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | data\clinical_notes\notes_example_1.txt | 43.000 | 0.000 | 31.200 | 136.000 | 84.000 | 4.300 | 0.600 | 102.000 | 52.000 | 65.000 | 144.000 | 0.960 | 82.000 | 198.000 | 15.100 | 7.900 | 41.000 | 221.000 | 268.000 | 136.000 | 0.000 | 1.000 | 0.000 | 0.000 | 1.000 | 1.000 | 1.000 | 1.000 | 4.000 | 1.401 | 0.657 | -0.641 | 0.800 | 8.400 | F2 | 1.400 | False | never | True | True | False | ['sedentary_work', 'family_history_liver_disease', 'clinical_signs_advanced_liver_disease'] |
| 2 | data\clinical_notes\notes_example_1.txt | 55.000 | 1.000 | 32.600 | 130.000 | 80.000 | 3.700 | 1.200 | 118.000 | 71.000 | 42.000 | 128.000 | 1.020 | 62.000 | 142.000 | 12.100 | 6.900 | 34.000 | 192.000 | 188.000 | 118.000 | 1.000 | 0.000 | 0.000 | 1.000 | 1.000 | 1.000 | 1.000 | 1.000 | 4.000 | 4.243 | 1.250 | 1.940 | 1.690 |  | F4 | 4.240 | True | former | False | True | True | ['sedentary_work', 'family_history_liver_disease', 'clinical_signs_advanced_liver_disease'] |


In [ ]:
note_cards = pd.read_csv(ARTIFACT_DIR / "clinical_note_demo_risk_cards.csv")
note_cards[[
    "note_index", "predicted_risk", "risk_score_0_100",
    "documented_lsm_kpa", "documented_fibrosis_stage",
    "documented_fib4_today", "smoking_status", "leakage_control",
]]


In [ ]:
for image_name in [
    "note_1_clinician_detailed_card.png",
    "note_1_xgboost_waterfall.png",
    "note_2_clinician_detailed_card.png",
    "note_2_xgboost_waterfall.png",
    "shap_xgboost_global_importance.png",
]:
    path = ARTIFACT_DIR / image_name
    print(path)
    try:
        from IPython.display import Image, display
        display(Image(filename=str(path)))
    except Exception:
        pass


## Final Methodological Notes

- NHANES is cross-sectional, so this is a feasibility proof of concept, not a prospective clinical model.
- External validation should use UK Biobank imaging/EHR and then temporal hospital EHR validation.
- The production version should add urine albumin/creatinine, longitudinal referral/care-gap data, and approved note abstraction fields.
- Calibration must be repeated before showing site-specific patient-facing risk percentages.
